In [3]:
import numpy as np
import pandas as pd
import doubleml as dml
import matplotlib.pyplot as plt # Plotting
import networkx as nx
from IPython.display import Image, display
from itertools import combinations
from sklearn.linear_model import LassoCV, Lasso
import re

In [24]:
psid_main = pd.read_csv('https://raw.githubusercontent.com/rudi-mac/causalml/main/psid_main.csv', index_col=0)
psid_main['Gender_Female'] = np.where(psid_main['Gender_Male']==1,0,1)
psid_main['Wealth_Quartiles'] = pd.qcut(psid_main['Wealth_abs'],4,labels=['Wealth_q1','Wealth_q2','Wealth_q3','Wealth_q4'])
# since you need a bachelor's degree to obtain a master's degree, we set all values of Bachelor_Degree to 1 if Masters_Degree is 1
psid_main.loc[psid_main['Masters_Degree'] == 1, 'Bachelor_Degree'] = 1
psid_main = psid_main[['Gender_Female','Ethnicity','Region','Born_Foreign','Hourly_Salary','Educ_Mother','Educ_Father','Study_Area','Childcare_Hrs','Bachelor_Degree','Wealth_Quartiles']]
mapping_mother = {1:'Mother_0-5_grades',2:'Mother_6-8_grades',3:'Mother_9-11_grades',4:'Mother_12_grades',5:'Mother_12_grades_plus',6:'Mother_Some_college',7:'Mother_College_BA',8:'Mother_College_advanced'}
mapping_father = {1:'Father_0-5_grades',2:'Father_6-8_grades',3:'Father_9-11_grades',4:'Father_12_grades',5:'Father_12_grades_plus',6:'Father_Some_college',7:'Father_College_BA',8:'Father_College_advanced'}
psid_main['Educ_Mother'] = psid_main['Educ_Mother'].replace(mapping_mother)
psid_main['Educ_Father'] = psid_main['Educ_Father'].replace(mapping_father)
psid_main['Bachelor_Degree'] = psid_main['Bachelor_Degree'].astype(int)
psid_main = psid_main.rename({'Wealth_Quartiles':'Family_Wealth','Bachelor_Degree':'Degree','Study_Area':'Area_of_Study'},axis=1)
psid_main

/var/folders/1v/zk8xltnj3kz6cs0mjz5hgsh00000gr/T/ipykernel_13691/2037286978.py:5: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  psid_main.loc[psid_main['Masters_Degree'] == 1, 'Bachelor_Degree'] = 1


,Gender_Female,Ethnicity,Region,Born_Foreign,Hourly_Salary,Educ_Mother,Educ_Father,Area_of_Study,Childcare_Hrs,Degree,Family_Wealth
0,1,White,South,0,2.500000,Mother_9-11_grades,Father_9-11_grades,Inap.,0.0,0,Wealth_q3
1,0,White,North Central,0,16.666667,Mother_9-11_grades,Father_12_grades,Inap.,0.0,0,Wealth_q1
2,0,White,South,0,16.500000,Mother_12_grades,Father_6-8_grades,Inap.,0.0,0,Wealth_q2
3,1,White,South,0,12.500000,Mother_12_grades,Father_12_grades,Inap.,140.0,0,Wealth_q3
4,0,White,South,0,24.960000,Mother_6-8_grades,Father_9-11_grades,Inap.,0.0,0,Wealth_q3
...,...,...,...,...,...,...,...,...,...,...,...
4674,1,Black,West,0,28.000000,Mother_12_grades,Father_Some_college,Health Professions and Related Programs,0.0,0,Wealth_q1
4675,1,Black,South,0,12.000000,Mother_12_grades,Father_12_grades,Inap.,0.0,0,Wealth_q1
4676,1,Black,South,0,19.750000,Mother_12_grades,Father_12_grades,Inap.,0.0,0,Wealth_q1
4677,1,Black,South,0,20.000000,Mother_9-11_grades,Father_12_grades,Inap.,20.0,0,Wealth_q2


In [25]:
psid_main.to_csv('sample_data_psid.csv')

In [30]:
# we need to convert categorical variables into one-hot encoded data
# Furthermore, we use the logarithm of salary to account for distribution skewness (e.g., https://kenbenoit.net/assets/courses/ME104/logmodels2.pdf#:~:text=URL%3A%20https%3A%2F%2Fkenbenoit.net%2Fassets%2Fcourses%2FME104%2Flogmodels2.pdf%0AVisible%3A%200%25%20)
psid_main_num = psid_main.copy()
# one-hot encoding of categorical variables
psid_main_num = pd.concat([psid_main_num, pd.get_dummies(psid_main.Ethnicity, dtype=int)], axis=1)
psid_main_num = pd.concat([psid_main_num, pd.get_dummies(psid_main.Region, dtype=int)], axis=1)
psid_main_num = pd.concat([psid_main_num, pd.get_dummies(psid_main.Educ_Father, dtype=int)], axis=1)
psid_main_num = psid_main_num.rename(columns={1:'Father_0-5_grades',2:'Father_6-8_grades',3:'Father_9-11_grades',4:'Father_12_grades',5:'Father_12_grades_plus',6:'Father_Some_college',7:'Father_College_BA',8:'Father_College_advanced'})
psid_main_num = pd.concat([psid_main_num, pd.get_dummies(psid_main.Educ_Mother, dtype=int)], axis=1)
psid_main_num = psid_main_num.rename(columns={1:'Mother_0-5_grades',2:'Mother_6-8_grades',3:'Mother_9-11_grades',4:'Mother_12_grades',5:'Mother_12_grades_plus',6:'Mother_Some_college',7:'Mother_College_BA',8:'Mother_College_advanced'})
psid_main_num = pd.concat([psid_main_num, pd.get_dummies(psid_main_num.Family_Wealth, dtype=int)], axis=1)

# remove one column per category to avoid "dummy trap"
#psid_main_num = psid_main_num.drop(['Other','South','Farming, Fishing, and Forestry Occupations','Mining, Quarrying, and Oil and Gas Extraction','Father_0-5_grades','Mother_0-5_grades','Wealth_Q1'], axis=1)

psid_main_num['Hourly_Salary_log'] = np.log(psid_main_num['Hourly_Salary'] )
psid_main_num['Gender_Female'] = psid_main_num['Gender_Female'].astype(int)
psid_main_num['Degree'] = psid_main_num['Degree'].astype(int)


In [31]:
psid_main_num

,Gender_Female,Ethnicity,Region,Born_Foreign,Hourly_Salary,Educ_Mother,Educ_Father,Area_of_Study,Childcare_Hrs,Degree,...,Mother_6-8_grades,Mother_9-11_grades,Mother_College_BA,Mother_College_advanced,Mother_Some_college,Wealth_q1,Wealth_q2,Wealth_q3,Wealth_q4,Hourly_Salary_log
0,1,White,South,0,2.500000,Mother_9-11_grades,Father_9-11_grades,Inap.,0.0,0,...,0,1,0,0,0,0,0,1,0,0.916291
1,0,White,North Central,0,16.666667,Mother_9-11_grades,Father_12_grades,Inap.,0.0,0,...,0,1,0,0,0,1,0,0,0,2.813411
2,0,White,South,0,16.500000,Mother_12_grades,Father_6-8_grades,Inap.,0.0,0,...,0,0,0,0,0,0,1,0,0,2.803360
3,1,White,South,0,12.500000,Mother_12_grades,Father_12_grades,Inap.,140.0,0,...,0,0,0,0,0,0,0,1,0,2.525729
4,0,White,South,0,24.960000,Mother_6-8_grades,Father_9-11_grades,Inap.,0.0,0,...,1,0,0,0,0,0,0,1,0,3.217275
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4674,1,Black,West,0,28.000000,Mother_12_grades,Father_Some_college,Health Professions and Related Programs,0.0,0,...,0,0,0,0,0,1,0,0,0,3.332205
4675,1,Black,South,0,12.000000,Mother_12_grades,Father_12_grades,Inap.,0.0,0,...,0,0,0,0,0,1,0,0,0,2.484907
4676,1,Black,South,0,19.750000,Mother_12_grades,Father_12_grades,Inap.,0.0,0,...,0,0,0,0,0,1,0,0,0,2.983153
4677,1,Black,South,0,20.000000,Mother_9-11_grades,Father_12_grades,Inap.,20.0,0,...,0,1,0,0,0,0,1,0,0,2.995732


In [33]:
# drop one category per group to avoid categorical dummy trap
# base category: male, white, no degree, US_born, Father_12_grades_plus, Mother_12_grades_plus, Wealth_q3, north central
columns = ['Gender_Female',
           'Asian', 'Black', 'Other', 'Native Hawaiian or Pacific Islander', #,'White'
           'Hourly_Salary_log',
           'Degree',
           #'Childcare_Hrs',
           'Born_Foreign',
           'Father_6-8_grades', 'Father_9-11_grades','Father_12_grades', 'Father_12_grades_plus', 'Father_Some_college','Father_College_BA', 'Father_College_advanced',#'Educ_Father',
           'Mother_6-8_grades', 'Mother_9-11_grades', 'Mother_12_grades','Mother_12_grades_plus', 'Mother_Some_college', 'Mother_College_BA','Mother_College_advanced',#'Educ_Mother',
           'Wealth_q1','Wealth_q2','Wealth_q4', #,'Wealth_q3','Wealth_abs',
           'South', 'Northeast', 'West', #,'North Central'
           #'Blue','Red' #,'Swing'
           ]

causal_df = psid_main_num[columns].copy()
causal_df = causal_df.loc[:,~causal_df.columns.duplicated()].copy()
causal_df = causal_df.reset_index(drop=True)
causal_df.rename({'West':'Region_West','Northeast':'Region_Northeast','South':'Region_South'},axis=1,inplace=True)
causal_df.rename({'Asian':'Ethnic_Asian','Black':'Ethnic_Black','Other':'Ethnic_Other','Native Hawaiian or Pacific Islander':'Ethnic_Hawaiian_Pac_Islander'},axis=1,inplace=True)

causal_df

,Gender_Female,Ethnic_Asian,Ethnic_Black,Ethnic_Other,Ethnic_Hawaiian_Pac_Islander,Hourly_Salary_log,Degree,Born_Foreign,Father_6-8_grades,Father_9-11_grades,...,Mother_12_grades_plus,Mother_Some_college,Mother_College_BA,Mother_College_advanced,Wealth_q1,Wealth_q2,Wealth_q4,Region_South,Region_Northeast,Region_West
0,1,0,0,0,0,0.916291,0,0,0,1,...,0,0,0,0,0,0,0,1,0,0
1,0,0,0,0,0,2.813411,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
2,0,0,0,0,0,2.803360,0,0,1,0,...,0,0,0,0,0,1,0,1,0,0
3,1,0,0,0,0,2.525729,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
4,0,0,0,0,0,3.217275,0,0,0,1,...,0,0,0,0,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4653,1,0,1,0,0,3.332205,0,0,0,0,...,0,0,0,0,1,0,0,0,0,1
4654,1,0,1,0,0,2.484907,0,0,0,0,...,0,0,0,0,1,0,0,1,0,0
4655,1,0,1,0,0,2.983153,0,0,0,0,...,0,0,0,0,1,0,0,1,0,0
4656,1,0,1,0,0,2.995732,0,0,0,0,...,0,0,0,0,0,1,0,1,0,0


In [34]:
# Generate all possible combinations of two-way interactions
unique_values = causal_df.drop('Hourly_Salary_log',axis=1).columns
all_combinations_2w = list(combinations(unique_values, 2))
print(len(all_combinations_2w))

351


In [35]:
interaction_terms_2way = []
for combi in all_combinations_2w:
    column_name = combi[0]+':'+ combi[1]

    if combi[0][:6] != combi[1][:6]:
        #print(column_name)
        causal_df[column_name] = causal_df[combi[0]]*causal_df[combi[1]]
        interaction_terms_2way.append(column_name)
causal_df = causal_df.copy()
causal_df.shape

/var/folders/1v/zk8xltnj3kz6cs0mjz5hgsh00000gr/T/ipykernel_13691/1267174352.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  causal_df[column_name] = causal_df[combi[0]]*causal_df[combi[1]]
/var/folders/1v/zk8xltnj3kz6cs0mjz5hgsh00000gr/T/ipykernel_13691/1267174352.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  causal_df[column_name] = causal_df[combi[0]]*causal_df[combi[1]]
/var/folders/1v/zk8xltnj3kz6cs0mjz5hgsh00000gr/T/ipykernel_13691/1267174352.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usual

(4658, 325)

In [36]:
# Generate all possible combinations of three-way interactions
all_combinations_3w = list(combinations(unique_values, 3))
print(len(all_combinations_3w))

2925


In [37]:
interaction_terms_3way = []
causal_df_3w = causal_df.copy()
for combi in all_combinations_3w:
    column_name = combi[0] + ':' + combi[1] + ':' + combi[2]

    if (combi[0][:6] != combi[1][:6]) & (combi[0][:6] != combi[2][:6]) & (combi[1][:6] != combi[2][:6]):
        #print(column_name)
        causal_df_3w[column_name] = causal_df[combi[0]]*causal_df[combi[1]]*causal_df[combi[2]]
        interaction_terms_3way.append(column_name)
causal_df_3w = causal_df_3w.copy()
causal_df_3w.shape

/var/folders/1v/zk8xltnj3kz6cs0mjz5hgsh00000gr/T/ipykernel_13691/865492700.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  causal_df_3w[column_name] = causal_df[combi[0]]*causal_df[combi[1]]*causal_df[combi[2]]
/var/folders/1v/zk8xltnj3kz6cs0mjz5hgsh00000gr/T/ipykernel_13691/865492700.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  causal_df_3w[column_name] = causal_df[combi[0]]*causal_df[combi[1]]*causal_df[combi[2]]
/var/folders/1v/zk8xltnj3kz6cs0mjz5hgsh00000gr/T/ipykernel_13691/865492700.py:8: PerformanceWarning: Dat

(4658, 2052)

In [39]:
ml_l = Lasso(fit_intercept=True, alpha=1) # Outcome Model: A machine learner for the nuisance function l0(X) = E[Y|X]
ml_m = Lasso(fit_intercept=True, alpha=1) # Treatment Model: A machine learner for the nuisance function m0(X) = E[D|X] 
ml_g = Lasso(fit_intercept=True) # A machine learner for the nuisance function g0(X) = E[Y-D*theta0|X]

par_grids = {'ml_l': {'alpha': np.arange(0.1, 1., 0.05)},
             'ml_m': {'alpha': np.arange(0.1, 1., 0.05)}}

In [40]:
# drop all columns that are constant / dont have any variation
causal_df_3w.drop('Ethnic_Hawaiian_Pac_Islander:Bachelor_Degree:Female', axis=1 ,inplace=True)

KeyError: "['Ethnic_Hawaiian_Pac_Islander:Bachelor_Degree:Female'] not found in axis"

In [45]:
# Construct treatment variables: 
# Main effects: Bachelor_Degree, Female
# 2-way interaction: Bachelor_Degree:Female  
# 3-way interactions: Bachelor_Degree:Female:X for various moderators X

treatment_variables = []

# Add main effects
treatment_variables.append('Degree')
treatment_variables.append('Gender_Female')

# Add 2-way interaction Bachelor_Degree:Female
if 'Degree:Gender_Female' in causal_df_3w.columns:
    treatment_variables.append('Degree:Gender_Female')
elif 'Female:Degree' in causal_df_3w.columns:
    treatment_variables.append('Gender_Female:Degree')
else:
    # Create it if it doesn't exist
    causal_df_3w['Degree:Gender_Female'] = causal_df_3w['Degree'] * causal_df_3w['Gender_Female']
    treatment_variables.append('Degree:Gender_Female')

# Add all 3-way interactions: Bachelor_Degree:Female:X
pattern1 = r'.*Gender_Female.*'
pattern2 = r'.*Degree.*'
for col in causal_df_3w.columns:
    if re.search(pattern1, col, re.IGNORECASE) and re.search(pattern2, col, re.IGNORECASE):
        # Check if it's a 3-way interaction (has 2 colons)
        if col.count(':') == 2:
            # Make sure it's not a duplicate and not already in the list
            if col not in treatment_variables:
                # Check that the column is not constant (has variation)
                if causal_df_3w[col].nunique() > 1:
                    treatment_variables.append(col)

print(f"Total treatment variables: {len(treatment_variables)}")
print(f"Main effects: Degree, Gender_Female")
print(f"2-way interaction: 1")
print(f"3-way interactions: {len(treatment_variables) - 3}")
print(f"\nTreatment variables: {treatment_variables}")

Total treatment variables: 27
Main effects: Degree, Gender_Female
2-way interaction: 1
3-way interactions: 24

Treatment variables: ['Degree', 'Gender_Female', 'Degree:Gender_Female', 'Gender_Female:Ethnic_Asian:Degree', 'Gender_Female:Ethnic_Black:Degree', 'Gender_Female:Ethnic_Other:Degree', 'Gender_Female:Degree:Born_Foreign', 'Gender_Female:Degree:Father_6-8_grades', 'Gender_Female:Degree:Father_9-11_grades', 'Gender_Female:Degree:Father_12_grades', 'Gender_Female:Degree:Father_12_grades_plus', 'Gender_Female:Degree:Father_Some_college', 'Gender_Female:Degree:Father_College_BA', 'Gender_Female:Degree:Father_College_advanced', 'Gender_Female:Degree:Mother_6-8_grades', 'Gender_Female:Degree:Mother_9-11_grades', 'Gender_Female:Degree:Mother_12_grades', 'Gender_Female:Degree:Mother_12_grades_plus', 'Gender_Female:Degree:Mother_Some_college', 'Gender_Female:Degree:Mother_College_BA', 'Gender_Female:Degree:Mother_College_advanced', 'Gender_Female:Degree:Wealth_q1', 'Gender_Female:Degree:

In [47]:
# Create DoubleMLData object with multiple treatments
# Y: log(Hourly_Salary)
# D: treatment variables (Bachelor_Degree, Female, and their interactions)
# X: all other covariates (use_other_treat_as_covariate=True means other treatments are also used as covariates)

print("Creating DoubleMLData object...")
print(f"Outcome variable: Hourly_Salary_log")
print(f"Treatment variables: {len(treatment_variables)} treatments")
print(f"All other columns will be used as covariates")

obj_dml_data = dml.DoubleMLData(causal_df_3w,
                                y_col='Hourly_Salary_log',
                                d_cols=treatment_variables,
                                use_other_treat_as_covariate=True)

print(f"\nDoubleMLData object created successfully!")
print(f"Number of observations: {obj_dml_data.n_obs}")
print(f"Number of covariates: {obj_dml_data.n_coefs}")

Creating DoubleMLData object...
Outcome variable: Hourly_Salary_log
Treatment variables: 27 treatments
All other columns will be used as covariates

DoubleMLData object created successfully!
Number of observations: 4658
Number of covariates: 27


In [48]:
# Create DoubleMLPLR object
# PLR = Partially Linear Regression model
# Model: Y = D*theta + g(X) + epsilon, where E[D|X] = m(X) + u
# We use Lasso for both nuisance functions (outcome and treatment models)

print("Creating DoubleMLPLR object...")
dml_plr = dml.DoubleMLPLR(obj_dml_data,
                          ml_l=ml_l,  # Outcome model learner
                          ml_m=ml_m,  # Treatment model learner  
                          ml_g=None,  # Optional: learner for g(X) = E[Y-D*theta|X]
                          n_folds=5,  # 5-fold cross-fitting
                          n_rep=5)    # 5 repetitions for stability

print("DoubleMLPLR object created successfully!")

Creating DoubleMLPLR object...
DoubleMLPLR object created successfully!


In [49]:
# Fit the DML model
print("Fitting DoubleMLPLR model...")
print("This may take a few minutes due to cross-fitting and multiple repetitions...")
dml_plr.fit()
print("\nModel fitted successfully!")

Fitting DoubleMLPLR model...
This may take a few minutes due to cross-fitting and multiple repetitions...


/Users/rudolfm/Documents - MTEC-SMI-322/Code/causalml-gui/venv/lib/python3.11/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/rudolfm/Documents - MTEC-SMI-322/Code/causalml-gui/venv/lib/python3.11/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/rudolfm/Documents - MTEC-SMI-322/Code/causalml-gui/venv/lib/python3.11/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/rudolfm/Documents - MTEC-SMI-322/Code/causalml-gui/venv/lib/python3.11/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/rudol


Model fitted successfully!


In [59]:
# Display comprehensive DML results
print("="*80)
print("DOUBLE MACHINE LEARNING RESULTS - COMPREHENSIVE REPORT")
print("="*80)

# Get the summary
summary_df = dml_plr.summary
print("\n" + "="*80)
print("COEFFICIENT ESTIMATES")
print("="*80)
print(summary_df)

# Get confidence intervals
print("\n" + "="*80)
print("CONFIDENCE INTERVALS (95%)")
print("="*80)
conf_int_95 = dml_plr.confint(level=0.95)
print(conf_int_95)

# Get p-values
print("\n" + "="*80)
print("HYPOTHESIS TESTS (H0: theta = 0)")
print("="*80)
pvals = dml_plr.pval
print(f"Treatment Variables and P-values:")
for i, treat in enumerate(treatment_variables):
    print(f"{treat:50s} p-value: {pvals[i]:.6f}")

# Model diagnostics
print("\n" + "="*80)
print("MODEL DIAGNOSTICS")
print("="*80)
learner_eval = dml_plr.evaluate_learners()
#print(f"\nOutcome Model (ml_l) RMSE: {learner_eval['ml_l'][0]:.6f}")
#print(f"Treatment Model (ml_m) RMSE:")
#for i, treat in enumerate(treatment_variables):
#    rmse = learner_eval['ml_m'][i] if hasattr(learner_eval['ml_m'], '__getitem__') else learner_eval['ml_m']
#    print(f"  {treat:50s} RMSE: {rmse:.6f}%")

# Sensitivity analysis parameters
print("\n" + "="*80)
print("SENSITIVITY ANALYSIS")
print("="*80)
if hasattr(dml_plr, 'sensitivity_params') and dml_plr.sensitivity_params is not None:
    rv_values = dml_plr.sensitivity_params.get('rv', None)
    if rv_values is not None:
        print("Robustness Values (RV) - proportion of residual variance explained by unobserved confounder:")
        for i, treat in enumerate(treatment_variables):
            rv = rv_values[i] if hasattr(rv_values, '__getitem__') else rv_values
            print(f"  {treat:50s} RV: {rv*100:.2f}%")

# Key findings summary
print("\n" + "="*80)
print("KEY FINDINGS SUMMARY")
print("="*80)
print("\nStatistically significant effects (p < 0.05):")
sig_effects = summary_df[summary_df['P>|t|'] < 0.05]
if len(sig_effects) > 0:
    for idx in sig_effects.index:
        coef = sig_effects.loc[idx, 'coef']
        se = sig_effects.loc[idx, 'std err']
        pval = sig_effects.loc[idx, 'P>|t|']
        print(f"\n{idx}:")
        print(f"  Coefficient: {coef:.4f} (SE: {se:.4f}, p-value: {pval:.6f})")
        if coef > 0:
            print(f"  Interpretation: Positive effect on log(Hourly_Salary)")
        else:
            print(f"  Interpretation: Negative effect on log(Hourly_Salary)")
else:
    print("No statistically significant effects at p < 0.05 level")

print("\n" + "="*80)

DOUBLE MACHINE LEARNING RESULTS - COMPREHENSIVE REPORT

COEFFICIENT ESTIMATES
                                                  coef   std err          t  \
Degree                                        0.451818  0.018255  24.750426   
Gender_Female                                -0.239860  0.017266 -13.892007   
Degree:Gender_Female                          0.182625  0.022735   8.032638   
Gender_Female:Ethnic_Asian:Degree             0.327874  0.010287  31.873838   
Gender_Female:Ethnic_Black:Degree             0.096474  0.036863   2.617066   
Gender_Female:Ethnic_Other:Degree             0.241711  0.173133   1.396102   
Gender_Female:Degree:Born_Foreign             0.323331  0.206764   1.563764   
Gender_Female:Degree:Father_6-8_grades        0.365582  0.190465   1.919413   
Gender_Female:Degree:Father_9-11_grades      -0.041594  0.084512  -0.492166   
Gender_Female:Degree:Father_12_grades         0.090526  0.043774   2.068046   
Gender_Female:Degree:Father_12_grades_plus    0.08130

In [61]:
# Create detailed results dataframe for export/further analysis
results_detailed = pd.DataFrame({
    'treatment': treatment_variables,
    'coefficient': dml_plr.coef,
    'std_error': dml_plr.se,
    't_statistic': dml_plr.coef / dml_plr.se,
    'p_value': dml_plr.pval,
})

# Add confidence intervals
ci_95 = dml_plr.confint(level=0.95)
results_detailed['ci_lower_95'] = ci_95.iloc[:, 0].values
results_detailed['ci_upper_95'] = ci_95.iloc[:, 1].values

ci_99 = dml_plr.confint(level=0.99)
results_detailed['ci_lower_99'] = ci_99.iloc[:, 0].values
results_detailed['ci_upper_99'] = ci_99.iloc[:, 1].values

# Add significance indicators
results_detailed['sig_1pct'] = results_detailed['p_value'] < 0.01
results_detailed['sig_5pct'] = results_detailed['p_value'] < 0.05
results_detailed['sig_10pct'] = results_detailed['p_value'] < 0.10

# Add model diagnostics
#learner_eval = dml_plr.evaluate_learners()
#results_detailed['outcome_model_rmse'] = learner_eval['ml_l'][0]
#if hasattr(learner_eval['ml_m'], '__getitem__'):
#    results_detailed['treatment_model_rmse'] = [learner_eval['ml_m'][i] for i in range(len(treatment_variables))]
#else:
#    results_detailed['treatment_model_rmse'] = learner_eval['ml_m']

# Sort by p-value
results_detailed = results_detailed.sort_values('p_value').reset_index(drop=True)

# Display the dataframe
print("\n" + "="*80)
print("DETAILED RESULTS DATAFRAME (sorted by p-value)")
print("="*80)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)
print(results_detailed)

# Export to CSV (optional)
# results_detailed.to_csv('dml_results_detailed.csv', index=False)
# print("\nResults exported to 'dml_results_detailed.csv'")


DETAILED RESULTS DATAFRAME (sorted by p-value)
                                       treatment  coefficient  std_error  \
0              Gender_Female:Ethnic_Asian:Degree     0.327874   0.010287   
1                                         Degree     0.451818   0.018255   
2                                  Gender_Female    -0.239860   0.017266   
3                 Gender_Female:Degree:Wealth_q4     0.352695   0.034257   
4                           Degree:Gender_Female     0.182625   0.022735   
5         Gender_Female:Degree:Father_College_BA     0.242441   0.037812   
6          Gender_Female:Degree:Region_Northeast     0.324347   0.051169   
7         Gender_Female:Degree:Mother_College_BA     0.203617   0.036488   
8   Gender_Female:Degree:Mother_College_advanced     0.229285   0.046354   
9   Gender_Female:Degree:Father_College_advanced     0.213910   0.049853   
10             Gender_Female:Degree:Region_South     0.125822   0.030082   
11      Gender_Female:Degree:Father_Some